# 📟 Stage 3 — On-Device Deployment & Measurement

**Project:** Energy-Aware Visual Anomaly Detection on MCUs — Stage 3 / 4

Takes the Stage 2 compressed models, converts them to TFLite INT8, deploys them on the **Arduino Nano 33 BLE Sense Rev 2**, and measures real latency, memory, and energy per inference.

## The two halves of Stage 3

**Half A — in Colab (this notebook):** PyTorch → ONNX → TFLite INT8 conversion, with numerical verification that the TFLite output matches PyTorch, and conversion of each model to a C header array for the Arduino.

**Half B — on the Arduino (C++ firmware, provided as text cells):** load a model, run inference on a stored test image, report inference latency and memory. Energy is measured externally with a current shunt / power profiler.

## What replaces what

The **simulated INT8** numbers from Stage 2 (theoretical size, weight-only fake-quant accuracy) are replaced here by **real measurements**: actual TFLite INT8 model size, measured on-device latency, measured SRAM/flash usage, and measured energy. The Stage 2 Pareto plot gets its size/energy axes updated with hardware truth.

## The conversion is the hard part

PyTorch → TFLite for conv-heavy models with transpose convolutions is historically fragile. We convert **one model end-to-end and verify it first**, then batch. If a model fails to convert, the notebook reports which one and why rather than silently producing a broken artifact.

---

# Zone 1 — Setup

## 1. Install conversion toolchain
Restart the runtime once if imports fail in the next cell (TF + ONNX sometimes need a fresh interpreter).

In [ ]:
!pip install -q onnx onnxruntime onnx2tf tensorflow ai-edge-litert
!pip install -q onnx-graphsurgeon sng4onnx onnxsim
print('Toolchain installed.')

## 2. Imports, Drive, config

In [ ]:
import os, json, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

from google.colab import drive
drive.mount('/content/drive')

# symlink data
src = '/content/drive/MyDrive/mvtec'; dst = '/content/mvtec'
if os.path.islink(dst): os.unlink(dst)
elif os.path.isdir(dst): shutil.rmtree(dst)
os.symlink(src, dst)

# pull Stage 2 models from Drive
STAGE2_DIR = Path('/content/stage2_out'); STAGE2_DIR.mkdir(exist_ok=True)
drive_s2 = Path('/content/drive/MyDrive/mvtec_stage2')
for f in drive_s2.glob('*'):
    shutil.copy(f, STAGE2_DIR / f.name)

OUT = Path('/content/stage3_out'); OUT.mkdir(exist_ok=True)
(OUT / 'tflite').mkdir(exist_ok=True)
(OUT / 'headers').mkdir(exist_ok=True)

CFG = {'data_root': '/content/mvtec', 'image_size': 128,
       'base_channels': 32, 'latent_channels': 32, 'n_down': 3, 'batch_size': 32}
PROTOCOL = {'border': 8, 'blur_kernel': 5, 'pool': 'mean'}

print('Stage 2 models available:')
for f in sorted(STAGE2_DIR.glob('model_*.pt')): print('  ', f.name)

## 3. Model + data (to rebuild and verify)

In [ ]:
def conv_block(in_c, out_c, stride=2):
    return nn.Sequential(nn.Conv2d(in_c, out_c, 4, stride, 1, bias=True), nn.ReLU(inplace=True))
def deconv_block(in_c, out_c, last=False):
    layers = [nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=True)]
    if not last: layers.append(nn.ReLU(inplace=True))
    return nn.Sequential(*layers)

class CompactAE(nn.Module):
    def __init__(self, base=32, latent=32, n_down=3):
        super().__init__()
        chans = [3] + [base*(2**i) for i in range(n_down)] + [latent]
        self.enc = nn.Sequential(*[conv_block(chans[i], chans[i+1]) for i in range(len(chans)-1)])
        rev = list(reversed(chans))
        self.dec = nn.Sequential(*[deconv_block(rev[i], rev[i+1], last=(i==len(rev)-2))
                                   for i in range(len(rev)-1)], nn.Sigmoid())
    def forward(self, x): return self.dec(self.enc(x))

def rebuild_from_checkpoint(ckpt_path):
    """Rebuild a (possibly pruned) CompactAE from its checkpoint, using the
    actual weight shapes in the state_dict so pruned architectures load correctly."""
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    sd = ckpt['state_dict']
    model = CompactAE(CFG['base_channels'], CFG['latent_channels'], CFG['n_down'])
    try:
        model.load_state_dict(sd); model.eval(); return model, ckpt
    except RuntimeError:
        pass  # pruned -> rebuild conv shapes from state_dict
    for i in range(len(model.enc)):
        w = sd[f'enc.{i}.0.weight']
        model.enc[i][0] = nn.Conv2d(w.shape[1], w.shape[0], 4, 2, 1, bias=True)
    for i in range(len(model.dec)-1):
        key = f'dec.{i}.0.weight'
        if key not in sd: continue
        w = sd[key]
        model.dec[i][0] = nn.ConvTranspose2d(w.shape[0], w.shape[1], 4, 2, 1, bias=True)
    model.load_state_dict(sd); model.eval()
    return model, ckpt

# Test dataset for calibration + verification
class MVTecTest(torch.utils.data.Dataset):
    def __init__(self, root, image_size):
        self.items = []
        for sub in sorted(Path(root, 'test').iterdir()):
            if not sub.is_dir(): continue
            label = 0 if sub.name == 'good' else 1
            for p in sorted(sub.rglob('*.png')): self.items.append((p, label, sub.name))
        self.tf = transforms.Compose([transforms.Resize(image_size+16),
                                      transforms.CenterCrop(image_size), transforms.ToTensor()])
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        p, l, d = self.items[i]
        return self.tf(Image.open(p).convert('RGB')), l, d

class MVTecTrain(torch.utils.data.Dataset):
    def __init__(self, root, image_size):
        self.paths = sorted(Path(root, 'train/good').rglob('*.png'))
        self.tf = transforms.Compose([transforms.Resize(image_size+16),
                                      transforms.CenterCrop(image_size), transforms.ToTensor()])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(Image.open(self.paths[i]).convert('RGB'))

# Zone 2 — Conversion (Half A)
*PyTorch → ONNX → TFLite INT8. Prove on one model, then batch.*

## 4. The converter

Path: PyTorch → ONNX (opset 13) → `onnx2tf` → TFLite with full INT8 quantization. Full INT8 needs a **representative dataset** (real images) so the converter can calibrate activation ranges — we feed defect-free training images. Input and output are kept INT8 for the smallest, fastest model on the MCU.

In [ ]:
import onnx, subprocess, tensorflow as tf

IMG = CFG['image_size']

def export_to_tflite(ckpt_path, name, rep_images, out_dir=OUT):
    """PyTorch checkpoint -> INT8 TFLite. Returns (tflite_path, size_kb) or raises."""
    model, ckpt = rebuild_from_checkpoint(ckpt_path)
    model.eval()

    # 1. PyTorch -> ONNX
    onnx_path = out_dir / f'{name}.onnx'
    dummy = torch.randn(1, 3, IMG, IMG)
    torch.onnx.export(model, dummy, str(onnx_path), opset_version=13,
                      input_names=['input'], output_names=['output'],
                      dynamic_axes=None)

    # 2. ONNX -> TF saved_model via onnx2tf
    tf_dir = out_dir / f'{name}_tf'
    if tf_dir.exists(): shutil.rmtree(tf_dir)
    r = subprocess.run(['onnx2tf', '-i', str(onnx_path), '-o', str(tf_dir), '-n'],
                       capture_output=True, text=True)
    if not (tf_dir / 'saved_model.pb').exists():
        raise RuntimeError(f'onnx2tf failed for {name}:\n{r.stderr[-1500:]}')

    # 3. TF -> TFLite INT8 with representative dataset
    def rep_gen():
        for img in rep_images:
            # onnx2tf converts to NHWC — provide NHWC float32 input
            x = img.permute(1, 2, 0).unsqueeze(0).numpy().astype(np.float32)
            yield [x]

    conv = tf.lite.TFLiteConverter.from_saved_model(str(tf_dir))
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = rep_gen
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    tflite_model = conv.convert()

    tflite_path = out_dir / 'tflite' / f'{name}.tflite'
    tflite_path.write_bytes(tflite_model)
    size_kb = len(tflite_model) / 1024
    return tflite_path, size_kb, ckpt

## 5. Convert ONE model end-to-end
Pick the trickiest case first (a pruned model with transpose convs). If this converts and verifies, the rest will.

In [ ]:
# representative dataset = a handful of defect-free training images
rep_ds = MVTecTrain(Path(CFG['data_root'])/'bottle', IMG)
rep_images = [rep_ds[i] for i in range(min(100, len(rep_ds)))]

TEST_NAME = 'model_bottle_Prune_50pct_l1_plus_INT8'
ckpt_file = STAGE2_DIR / f'{TEST_NAME}.pt'
assert ckpt_file.exists(), f'Not found: {ckpt_file}. Check Stage 2 saved it.'

tflite_path, size_kb, ckpt = export_to_tflite(ckpt_file, TEST_NAME, rep_images)
print(f'✓ Converted {TEST_NAME}')
print(f'  TFLite size: {size_kb:.1f} KB  (Stage 2 estimated: {ckpt["n_params"]/1024:.1f} KB)')

## 6. Verify: TFLite output ≈ PyTorch output
Critical step. A model that converts but produces wrong numbers is worse than one that fails loudly. We compare anomaly scores from the original PyTorch model vs the TFLite model on the test set, and check the AUROC is preserved.

In [ ]:
from ai_edge_litert.interpreter import Interpreter

def score_pytorch(model, test_ds):
    model.eval(); b = PROTOCOL['border']; k = PROTOCOL['blur_kernel']
    scores, labels = [], []
    with torch.no_grad():
        for x, y, d in DataLoader(test_ds, batch_size=16):
            err = (x - model(x)).pow(2).mean(1)
            err = err[:, b:-b, b:-b]
            err = F.avg_pool2d(err.unsqueeze(1), k, 1, k//2).squeeze(1)
            scores += err.flatten(1).mean(1).tolist(); labels += y.tolist()
    return np.array(scores), np.array(labels)

def score_tflite(tflite_path, test_ds):
    interp = Interpreter(model_path=str(tflite_path)); interp.allocate_tensors()
    inp = interp.get_input_details()[0]; out = interp.get_output_details()[0]
    in_scale, in_zp = inp['quantization']; out_scale, out_zp = out['quantization']
    b = PROTOCOL['border']; k = PROTOCOL['blur_kernel']
    scores, labels = [], []
    for x, y, d in test_ds:
        xn = x.permute(1,2,0).numpy()                       # HWC float [0,1]
        xq = np.round(xn/in_scale + in_zp).clip(-128,127).astype(np.int8)[None]
        interp.set_tensor(inp['index'], xq); interp.invoke()
        yq = interp.get_tensor(out['index'])[0].astype(np.float32)
        yhat = (yq - out_zp) * out_scale                     # dequant -> HWC float
        yhat = torch.tensor(yhat).permute(2,0,1)
        err = (x - yhat).pow(2).mean(0)[b:-b, b:-b]
        err = F.avg_pool2d(err[None,None], k, 1, k//2)[0,0]
        scores.append(err.mean().item()); labels.append(y)
    return np.array(scores), np.array(labels)

test_ds = MVTecTest(Path(CFG['data_root'])/'bottle', IMG)
model, _ = rebuild_from_checkpoint(ckpt_file)

s_pt, l_pt = score_pytorch(model, test_ds)
s_tf, l_tf = score_tflite(tflite_path, test_ds)
print(f'PyTorch AUROC : {roc_auc_score(l_pt, s_pt):.4f}')
print(f'TFLite  AUROC : {roc_auc_score(l_tf, s_tf):.4f}')
print(f'Score corr    : {np.corrcoef(s_pt, s_tf)[0,1]:.4f}  (want > 0.95)')

## 7. TFLite → C header for Arduino
`xxd`-style conversion: the .tflite bytes become a `const unsigned char[]` the firmware compiles in.

In [ ]:
def tflite_to_header(tflite_path, var_name, out_dir=OUT/'headers'):
    data = Path(tflite_path).read_bytes()
    lines = [f'// Auto-generated from {Path(tflite_path).name}',
             f'// {len(data)} bytes', '#pragma once', '',
             f'const unsigned int {var_name}_len = {len(data)};',
             f'alignas(8) const unsigned char {var_name}[] = {{']
    for i in range(0, len(data), 12):
        chunk = ', '.join(f'0x{b:02x}' for b in data[i:i+12])
        lines.append(f'  {chunk},')
    lines.append('};')
    hpath = out_dir / f'{var_name}.h'
    hpath.write_text('\n'.join(lines))
    return hpath, len(data)

hpath, nbytes = tflite_to_header(tflite_path, 'model_bottle_prune50')
print(f'✓ Header: {hpath.name}  ({nbytes/1024:.1f} KB)')

## 8. Batch-convert all deployment candidates
Run only after the single-model path above verifies. Converts each selected model, records real TFLite size, verifies AUROC, generates the header.

In [ ]:
# Deployment set (the defensible Pareto-optimal + reference picks)
DEPLOY = {
    'bottle': [
        # source checkpoint (FP32/pruned/distilled) → TFLite converter applies real INT8
        'model_bottle_Baseline_FP32',        # uncompressed reference (0.929)
        'model_bottle_Prune_50pct_l1',       # pruning leader on bottle (0.905)
        'model_bottle_Distill_b16',          # smallest (0.884)
    ],
    'hazelnut': [
        'model_hazelnut_Baseline_FP32',      # uncompressed reference (0.815)
        'model_hazelnut_Distill_b16',        # leader on hazelnut this run (0.857)
        'model_hazelnut_Prune_50pct_taylor', # best pruning (0.835), different technique
    ],
}

results = []
for cat, names in DEPLOY.items():
    rep_ds = MVTecTrain(Path(CFG['data_root'])/cat, IMG)
    rep_images = [rep_ds[i] for i in range(min(100, len(rep_ds)))]
    test_ds = MVTecTest(Path(CFG['data_root'])/cat, IMG)
    for name in names:
        ckpt_file = STAGE2_DIR / f'{name}.pt'
        if not ckpt_file.exists():
            print(f'⚠️  missing {name}, skipping'); continue
        try:
            tfl, size_kb, ckpt = export_to_tflite(ckpt_file, name, rep_images)
            model, _ = rebuild_from_checkpoint(ckpt_file)
            s_tf, l_tf = score_tflite(tfl, test_ds)
            auroc_tf = roc_auc_score(l_tf, s_tf)
            var = name.replace('model_', '').replace('_plus', '').replace('_sim','')
            tflite_to_header(tfl, var)
            results.append({'category': cat, 'name': name, 'tflite_kb': round(size_kb,1),
                            'auroc_tflite': round(auroc_tf,4),
                            'auroc_stage2': round(ckpt['auroc'],4), 'macs': ckpt['macs']})
            print(f'✓ {name}: {size_kb:.1f} KB, AUROC {auroc_tf:.4f}')
        except Exception as e:
            print(f'✗ {name} FAILED: {str(e)[:200]}')

import pandas as pd
df = pd.DataFrame(results)
df.to_csv(OUT / 'tflite_results.csv', index=False)
print(); print(df.to_string())

## 9. Export a test image as C array (for on-device inference)
The Arduino needs one stored image to run inference on. We export a defect and a good image, INT8-quantized to match the model input.

In [ ]:
def image_to_header(img_tensor, var_name, in_scale=1/255., in_zp=-128, out_dir=OUT/'headers'):
    x = img_tensor.permute(1,2,0).numpy()  # HWC float [0,1]
    xq = np.round(x/in_scale + in_zp).clip(-128,127).astype(np.int8).flatten()
    lines = [f'// test image {var_name}, INT8, {len(xq)} bytes (HWC {img_tensor.shape})',
             '#pragma once', f'const unsigned int {var_name}_len = {len(xq)};',
             f'const signed char {var_name}[] = {{']
    for i in range(0, len(xq), 12):
        lines.append('  ' + ', '.join(str(int(b)) for b in xq[i:i+12]) + ',')
    lines.append('};')
    p = out_dir / f'{var_name}.h'; p.write_text('\n'.join(lines)); return p

test_ds = MVTecTest(Path(CFG['data_root'])/'bottle', IMG)
good_i = next(i for i,(_,l,_) in enumerate(test_ds.items) if l==0)
bad_i  = next(i for i,(_,l,_) in enumerate(test_ds.items) if l==1)
image_to_header(test_ds[good_i][0], 'test_img_good')
image_to_header(test_ds[bad_i][0],  'test_img_defect')
print('✓ Exported test_img_good.h, test_img_defect.h')

## 10. Package headers + back up to Drive

In [ ]:
shutil.make_archive('/content/stage3_headers', 'zip', OUT/'headers')
shutil.copytree(OUT, '/content/drive/MyDrive/mvtec_stage3', dirs_exist_ok=True)
shutil.copy('/content/stage3_headers.zip', '/content/drive/MyDrive/mvtec_stage3/')
print('Headers + TFLite models backed up to Drive/mvtec_stage3')
print('Download stage3_headers.zip for the Arduino project.')
from google.colab import files
files.download('/content/stage3_headers.zip')

# Zone 3 — On-Device (Half B)
*C++ firmware for the Arduino. These are reference cells — copy into the Arduino IDE, not run in Colab.*

## 11. Arduino setup

**Board:** Arduino Nano 33 BLE Sense Rev 2.  
**Library:** install `Arduino_TensorFlowLite` (or `Chirale_TensorFlowLite` / `Harvard_TinyMLx` depending on availability in 2026) via the Library Manager.

**Project files** (place in one sketch folder):
- `stage3_firmware.ino` — the firmware below
- `model_bottle_prune50.h` — a converted model header (start with one)
- `test_img_good.h`, `test_img_defect.h` — the test images

The model arena size (`kTensorArenaSize`) must be large enough to hold the model's activations. Start at 128 KB and shrink once you see the reported peak usage. The Nano 33 BLE has 256 KB SRAM total.

## 12. Firmware — inference + latency + memory

```cpp
#include <TensorFlowLite.h>
#include "tensorflow/lite/micro/all_ops_resolver.h"
#include "tensorflow/lite/micro/micro_interpreter.h"
#include "tensorflow/lite/schema/schema_generated.h"

#include "model_bottle_prune50.h"   // the model C array
#include "test_img_good.h"          // stored test image (INT8)
#include "test_img_defect.h"

// Tensor arena — holds model activations. Tune down after first run.
constexpr int kTensorArenaSize = 130 * 1024;
alignas(16) uint8_t tensor_arena[kTensorArenaSize];

const tflite::Model* model = nullptr;
tflite::MicroInterpreter* interpreter = nullptr;
TfLiteTensor* input = nullptr;
TfLiteTensor* output = nullptr;

// Compute mean reconstruction error (the anomaly score) over the image,
// matching the frozen scoring protocol (minus border crop / blur, which
// can be added on-device or applied host-side from the raw error).
float reconstruction_error(const signed char* in, const signed char* out,
                           float in_scale, int in_zp,
                           float out_scale, int out_zp, int n) {
  double acc = 0.0;
  for (int i = 0; i < n; i++) {
    float a = (in[i]  - in_zp)  * in_scale;
    float b = (out[i] - out_zp) * out_scale;
    acc += (double)(a - b) * (a - b);
  }
  return (float)(acc / n);
}

void setup() {
  Serial.begin(115200);
  while (!Serial);

  model = tflite::GetModel(model_bottle_prune50);
  if (model->version() != TFLITE_SCHEMA_VERSION) {
    Serial.println("Model schema mismatch!"); while (1);
  }

  static tflite::AllOpsResolver resolver;
  static tflite::MicroInterpreter static_interpreter(
      model, resolver, tensor_arena, kTensorArenaSize);
  interpreter = &static_interpreter;

  if (interpreter->AllocateTensors() != kTfLiteOk) {
    Serial.println("AllocateTensors failed — increase arena size"); while (1);
  }

  input  = interpreter->input(0);
  output = interpreter->output(0);

  Serial.print("Arena used (bytes): ");
  Serial.println(interpreter->arena_used_bytes());
  Serial.print("Input bytes: ");  Serial.println(input->bytes);
  Serial.print("Output bytes: "); Serial.println(output->bytes);
}

void run_one(const signed char* img, int img_len, const char* label) {
  // copy stored image into the input tensor
  for (int i = 0; i < img_len; i++) input->data.int8[i] = img[i];

  unsigned long t0 = micros();
  interpreter->Invoke();
  unsigned long t1 = micros();

  float score = reconstruction_error(
      input->data.int8, output->data.int8,
      input->params.scale, input->params.zero_point,
      output->params.scale, output->params.zero_point, img_len);

  Serial.print(label); Serial.print("  score="); Serial.print(score, 6);
  Serial.print("  latency_us="); Serial.println(t1 - t0);
}

void loop() {
  run_one(test_img_good,   test_img_good_len,   "GOOD  ");
  run_one(test_img_defect, test_img_defect_len, "DEFECT");
  delay(2000);
}
```

## 13. What to record on-device

For each deployed model, from the serial output and external instruments:

| Metric | Source |
|---|---|
| **Latency (µs)** | `micros()` around `Invoke()` — printed by firmware |
| **Arena / SRAM (bytes)** | `interpreter->arena_used_bytes()` — printed at setup |
| **Flash (bytes)** | model header size + sketch size (Arduino IDE compile output) |
| **Energy (mJ/inference)** | external: current shunt + scope, or Nordic PPK II. Measure current during `Invoke()`, integrate over latency, subtract idle baseline |
| **Score separation** | GOOD vs DEFECT scores — confirms the model works on-device |

Average latency/energy over many inferences (e.g., 100 loops) and report mean ± std.

## 14. Energy measurement note

USB power gives no energy variation — fine for measuring *per-inference cost*, which is what Stage 3 needs. The *adaptive* energy behaviour is Stage 4. For per-inference energy:

1. Power the board through a known shunt resistor (e.g. 1 Ω) in the supply line.
2. Measure voltage across the shunt on a scope during `Invoke()`.
3. Current = V_shunt / R; power = current × supply voltage; energy = power × latency.
4. Subtract the idle (between-inference) baseline to isolate inference energy.

Or use a Nordic Power Profiler Kit II for a turnkey current trace.

# Stage 3 — Report & Stage 4 Handoff

## What Stage 3 produces

- TFLite INT8 models for each deployment candidate, with **verified** AUROC (TFLite vs PyTorch) and **real** model size (replaces Stage 2's theoretical estimate).
- On-device measurements: latency, SRAM (arena), flash, and energy per inference.
- The Stage 2 Pareto plot, re-drawn with **measured** size/energy on the cost axis instead of estimated.

## Caveats

- **Conversion can change AUROC slightly.** Full INT8 (activations too, not just weights as in Stage 2's simulation) may shift AUROC up or down by 1–2 points. The verified TFLite AUROC is the real number; update the report.
- **Arena size is model-specific.** Report the measured `arena_used_bytes` per model — it's part of the deployability story (must fit 256 KB SRAM).
- **Latency may not track MACs linearly.** Memory access and layer types matter. If a low-MACs model is slow, that's a finding.

## Stage 4 — handoff

Stage 4 (energy-aware adaptation) needs:

1. **The measured energy per inference** for each model — this defines the cost of each rung on the quality ladder.
2. **The accuracy of each model** (verified TFLite AUROC) — the benefit of each rung.
3. **All selected models co-resident in flash** — the firmware must hold several models and switch between them at runtime.

The Stage 4 adaptive policy will pick, per inference, which model to run based on remaining energy budget, trading accuracy for energy as the (simulated) battery depletes.

### Open questions for Stage 4
- How many models can co-reside in 1 MB flash simultaneously? (Sum of header sizes.)
- What energy-budget thresholds map to which model? (Needs the measured per-inference energies.)
- Threshold policy vs utility policy — implement both, compare.